# 🍄 Mario RL — RAM Training (PPO + MlpPolicy)

Trains a PPO agent using **RAM observations** from Stable-Retro. In this Mario integration, the RAM observation shape prints as `(10240,)`.

| Setting | Value |
|---|---|
| Algorithm | **PPO** (Proximal Policy Optimization) |
| Policy network | **MlpPolicy** (256×256 feedforward) |
| Observation | Stable-Retro RAM vector → float32, usually shape `(10240,)` |
| Level | World 1-1 (flag-locked) |
| Episode ends | Mario dies **or** reaches the flag |
| Training device | **auto** — GPU used for tensor ops if available; emulators always run on CPU |

> **Device note:** The 16 NES emulators always run on CPU (they're C++ processes).
> PPO's neural network — forward passes during rollout collection and gradient
> updates — runs on GPU if available. For a small MLP the speedup is modest but
> real. `--device auto` lets SB3 pick the best device automatically.
>
> **Runtime:** A GPU runtime (T4 / A100) is recommended.

## 1. Mount Google Drive
All checkpoints and logs are saved to Drive so they survive runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Adjust these paths to match your Drive layout
DRIVE_ROOT = '/content/drive/MyDrive/mario_rl'
MODEL_DIR  = f'{DRIVE_ROOT}/models'
LOG_DIR    = f'{DRIVE_ROOT}/runs'
ROM_DIR = '/content/mario-rl-ram/roms'
VIDEO_DIR  = f'{DRIVE_ROOT}/videos/ram'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)
print('Drive mounted. Directories ready.')

## 2. Clone Repo & Install

In [ ]:
import os
if not os.path.isdir('/content/mario-rl-ram'):
    !git clone https://github.com/hbofz/mario-rl-ram.git /content/mario-rl-ram
else:
    !git -C /content/mario-rl-ram pull

%cd /content/mario-rl-ram
!pip install -e . -q
print('Install complete. Restart runtime if prompted.')

## 3. Check Device
SB3 will use GPU for PPO's tensor ops (action selection + gradient updates) if CUDA is available.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('   SB3 will use GPU for MLP forward/backward passes.')
else:
    print('ℹ️  No GPU found — SB3 will use CPU. RAM+MLP training still works fine on CPU.')

print()
print('Note: regardless of GPU, the 16 NES emulators always run on CPU.')

# '--device auto' lets SB3 pick CUDA if available, otherwise CPU
DEVICE = 'auto'

## 4. Import ROM

Imports the ROM from the repo's `roms/` folder. If you keep your ROM in Drive instead, change `ROM_DIR` above.

In [ ]:
!python -m stable_retro.import {ROM_DIR}

## 5. Smoke Test
Quick sanity check — should print a RAM observation shape, usually `(10240,)`, and the action space.

In [ ]:
!mario-smoke --state Level1-1 --obs-mode ram --steps 300 --reward-mode base

## 6. Training Configuration
Edit the values below, then run **Section 7** to start training.

In [ ]:
# ── Training hyperparameters ──────────────────────────────────────────────────
RUN_NAME      = 'ram-1-1'     # name of this run (also the checkpoint subdirectory)
TOTAL_STEPS   = 10_000_000   # total environment steps
N_ENVS        = 16            # parallel emulator workers
N_STEPS       = 512           # steps per env per rollout
BATCH_SIZE    = 2048          # PPO minibatch size
N_EPOCHS      = 4             # PPO update epochs per rollout
LEARNING_RATE = 2.5e-4        # Adam learning rate
GAMMA         = 0.995         # discount factor
GAE_LAMBDA    = 0.95          # GAE lambda
ENT_COEF      = 0.01          # entropy coefficient (exploration)
CLIP_RANGE    = 0.2           # PPO clip range
CKPT_FREQ     = 250_000       # save a checkpoint every N steps

# ── Environment settings ──────────────────────────────────────────────────────
STATE         = 'Level1-1'    # which level to train on
OBS_MODE      = 'ram'         # 'ram' for this notebook
ACTION_MODE   = 'mario'       # curated 11-action discrete set
REWARD_MODE   = 'smart'       # per-level reward profile
ACTION_REPEAT = 4             # repeat each action for N emulator frames

print(f'Run:       {RUN_NAME}')
print(f'Algorithm: PPO  |  Policy: MlpPolicy (RAM observations)')
print(f'Device:    {DEVICE}  (SB3 auto-selects GPU if available)')
print(f'Level:     {STATE}  |  Steps: {TOTAL_STEPS:,}')
print(f'Envs: {N_ENVS}  |  Batch: {BATCH_SIZE}  |  Checkpoint every: {CKPT_FREQ:,} steps')

## 7. Train

**Auto-resume is on by default.** If Colab disconnects, just re-run this cell — the script
scans the checkpoint directory and automatically continues from the latest `.zip`.

In [ ]:
!mario-train \
  --obs-mode        {OBS_MODE} \
  --state           {STATE} \
  --flag-lock \
  --single-life \
  --algo            ppo \
  --timesteps       {TOTAL_STEPS} \
  --n-envs          {N_ENVS} \
  --n-steps         {N_STEPS} \
  --batch-size      {BATCH_SIZE} \
  --n-epochs        {N_EPOCHS} \
  --learning-rate   {LEARNING_RATE} \
  --gamma           {GAMMA} \
  --gae-lambda      {GAE_LAMBDA} \
  --ent-coef        {ENT_COEF} \
  --clip-range      {CLIP_RANGE} \
  --checkpoint-freq {CKPT_FREQ} \
  --action-mode     {ACTION_MODE} \
  --reward-mode     {REWARD_MODE} \
  --action-repeat   {ACTION_REPEAT} \
  --run-name        {RUN_NAME} \
  --model-dir       {MODEL_DIR} \
  --log-dir         {LOG_DIR} \
  --device          {DEVICE}

## 8. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## 9. Evaluate Final Model & Record Videos

In [ ]:
EVAL_MODEL    = f'{MODEL_DIR}/{RUN_NAME}/final_model.zip'
EVAL_EPISODES = 5

!mario-eval \
  --model     {EVAL_MODEL} \
  --obs-mode  {OBS_MODE} \
  --state     {STATE} \
  --episodes  {EVAL_EPISODES} \
  --video-dir {VIDEO_DIR} \
  --no-single-life \
  --no-flag-lock

## 10. Evaluate a Specific Checkpoint
Compare early vs. late training to see how the policy evolved.

In [ ]:
CHECKPOINT = f'{MODEL_DIR}/{RUN_NAME}/ppo_5000000_steps.zip'  # ← change step count

!mario-eval \
  --model     {CHECKPOINT} \
  --obs-mode  {OBS_MODE} \
  --state     {STATE} \
  --episodes  3 \
  --video-dir {VIDEO_DIR}/checkpoint_5M \
  --no-single-life \
  --no-flag-lock

## 11. List Saved Checkpoints

In [ ]:
import os, glob
ckpts = sorted(glob.glob(f'{MODEL_DIR}/{RUN_NAME}/*.zip'))
print(f'Found {len(ckpts)} checkpoint(s) in {MODEL_DIR}/{RUN_NAME}:')
for c in ckpts:
    size_mb = os.path.getsize(c) / 1e6
    print(f'  {os.path.basename(c)}  ({size_mb:.1f} MB)')